In [17]:
import numpy as np
import random
import os
import torch

from torch.utils.data import DataLoader
from torchvision import transforms

## seeding for reproducibility in random events like dropout, Weight initialization, Data shuffling

def seed_all(seed =42):
    torch.manual_seed(seed) #seeding in CPU
    np.random.seed(seed) #seeding in np lib operations
    if torch.cuda.is_available(): # if the gpu is available
        torch.cuda.manual_seed_all(seed) # seed those operations

seed_all(42)

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #initializing the device

print("using device:", device)

# configuration for architecture

n_qubits = 4
batch_size = 64 # higher as the size of image is very small
num_classes = 10
num_epochs = 50
lr = 0.0005

# small batchsize gives better generalization but a higher gives smooth gradient

using device: cuda


In [19]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Grayscale(1),
    #transforms.RandomHorizontalFlip(),  # check the effect of this, as its not recommended to use
    transforms.RandomRotation(10),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

test_transform = transforms.Compose([

    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
    
])

In [20]:
TRAIN_PATH = 'train'
TEST_PATH = 'test'

from torchvision.datasets import ImageFolder

try:
    # creates dataset object and label each classes
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    test_dataset = ImageFolder(TEST_PATH, transform=test_transform)

    print("dataset loaded successfully")

except Exception as e:
    print(f"Error loading datasets: {e}")

dataset loaded successfully


In [21]:
# giving each class weight as per it's size

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight = 'balanced', classes = np.unique(labels), y = labels) # array of class weights
    # converting to tensor for scikitlearn computation and giving it to the device used
    class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device) 
except:
    print("could not calculate class weights")
    print("Now using weights while training")
    class_weight_tensor = torch.ones(num_classes).to(device)

could not calculate class weights
Now using weights while training


In [22]:
from torch.utils.data import DataLoader

# after creating classes, shuffle dataset and group into batches
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True, num_workers=4,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False, num_workers=4, pin_memory=True)


In [23]:
import pennylane as qml

dev = qml.device("default.qubit",  wires = n_qubits) # device to run quantum cricuit

In [24]:
# to tasnform normal python function to qnode
    
# interface = torch - makes circuit compatible with PyTorch tensors and gradient will integrate with pytorch
# diff method - how gradients are calculated
# backfrop - uses classical automatic differentiator and only works with "default.qubit"

#weight(no_of samples, no_of_qubits=no_of_features)
@qml.qnode(dev, interface = "torch", diff_method = "backprop") # experiment with Parameter-shift also to validate its implication in real quantum device
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[...,i], wires = i) # maps parameter1....parameter_n for each sample 'required to corrospond each qubits with all the values of that feature'
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i) 
        for i in range(n_qubits -1):
            qml.CNOT(wires= [i,i+1]) #giving CNOT is very important to make qubits dependent of each other
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

## alternatives: RX, RZ, RX+RZ gates, ring topology(tried in malimg), full connectivity --- experiment with this in datasets
weight_shapes = {"weights": (2, n_qubits)} 

In [25]:
import torch.nn as nn

# creatingn a custom neural network to reduce the dimension of the input(before this analyze input properly)

class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0): #subjected to changes based on the performace
        super().__init__()
        self.conv = nn.Sequential(
            #conv2d(input_channel, output_channel, kernel_size, stride, padding)
            nn.Conv2d(1, 16, 3, stride=1, padding=1),
            nn.BatchNorm2d(16), # for channelwise normalization
            nn.ReLU(), #default,  if most cells are giving output 0- use LeakyRELU()
            nn.Dropout(dropout),# no need to use dropout, will use if required later

            nn.Conv2d(16, 32, 3, stride=2, padding=1), # if stride = 1 -> image will be average of 32 values in a single layer, cant focus on small patterns
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Conv2d(32, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.AdaptiveAvgPool2d((1,1))
            
        )
        #expand plus project strategy for better expessibility but adds risk of overfitting
        '''self.fc_expand = nn.Linear(32, final_dim*2)
        self.fc_project = nn.Linear(final_dim*2, final_dim)'''
        self.fc = nn.Linear(64, final_dim) # as dataset seems not to be that complex

    def forward(self, x):
        x = self.conv(x) # input must pass through all the layers defined earlier
        x= x.view(x.size(0), -1) 
        return self.fc(x)
        

In [26]:
class HybrideQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 32),  # projects data into 32 values
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes) 
        )

    def forward(self,x):
        x=self.feature_extractor(x)
        x=torch.tanh(x) # clips angle values to (-1,1)
        q_out=self.q_layer(x)
        return self.classifier(q_out)

In [27]:
from tqdm import tqdm

def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct = 0.0,0

    for inputs, labels in tqdm(dataloader, desc = "Training", leave = False):
        inputs, labels = inputs.to(device), labels.to(device) # giving input and its labels to the device for evaluation

        optimizer.zero_grad() # clears previous gradients(pytorch accumulate gradient of previous batch also)
        outputs = model(inputs) # gives prediction of the batch
        loss = loss_fn(outputs, labels) # finds loss by doing softmax(prob_of_correct_class)
        loss.backward() #computes gradients using backpropogation

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
        optimizer.step() # weights are updated: (w_new = w_old - lr*gradient)

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item() # no of total correct predictions given by network
        
    return total_loss/len(dataloader), correct/len(dataloader.dataset) #gives: average loss per sample, average correct predicted samples


In [28]:
eval_transform = transforms.Compose([

    transforms.Grayscale(1),
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
    
])

Val_path = './val'

val_dataset = ImageFolder(Val_path, transform= eval_transform)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, num_workers = 4, pin_memory=True)



In [29]:
def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs =model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1) #torch.max() gives (max_vlaue, index), so this will give intex of the maximum value(max probability class)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss/len(dataloader), correct/total

In [30]:
print("Initializing model...")

Initializing model...


In [31]:
from pennylane.qnn import TorchLayer

model = HybrideQNN(n_qubits= n_qubits, num_classes= num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)
loss_fn = loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor)
best_val_loss = float('inf')

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.5, patience =5)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

early_stopping_patience = 15

epochs_without_improvement = 0

print(f"Starting Training for {num_epochs} epochs with Two-Stage Loss Schedule...")

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    scheduler.step(val_loss)

     
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "exp1_1.pth")
        print("   💾 Best model saved.")
    else:
        epochs_without_improvement += 1
        print(f"   🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"⏹️ Early stopping triggered after {epoch+1} epochs.")
        break

Starting Training for 50 epochs with Two-Stage Loss Schedule...


Epoch 1/50
   Train Loss: 1.6045 | Train Acc: 0.4671 | Val Acc: 0.6946
   💾 Best model saved.


Epoch 2/50
   Train Loss: 0.9244 | Train Acc: 0.7047 | Val Acc: 0.7226
   💾 Best model saved.


Epoch 3/50
   Train Loss: 0.7976 | Train Acc: 0.7233 | Val Acc: 0.7310
   💾 Best model saved.


Epoch 4/50
   Train Loss: 0.7515 | Train Acc: 0.7333 | Val Acc: 0.7575
   💾 Best model saved.


...

Epoch 46/50
   Train Loss: 0.4025 | Train Acc: 0.8728 | Val Acc: 0.8780
   💾 Best model saved.


Epoch 47/50
   Train Loss: 0.3984 | Train Acc: 0.8731 | Val Acc: 0.8772
   🕒 No improvement for 1 epoch(s).


Epoch 48/50
   Train Loss: 0.3940 | Train Acc: 0.8754 | Val Acc: 0.8751
   🕒 No improvement for 2 epoch(s).


Epoch 49/50
   Train Loss: 0.3947 | Train Acc: 0.8744 | Val Acc: 0.8805
   🕒 No improvement for 3 epoch(s).


Epoch 50/50
   Train Loss: 0.3958 | Train Acc: 0.8747 | Val Acc: 0.8834
   🕒 No improvement for 4 epoch(s).
